In [ ]:
# imports
import sys, os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), os.pardir)))
from src.mnist_mlp import MLP
from src import *

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import itertools, json

In [ ]:
# TODO: Plot validation loss in reference to number of params and number of hidden layers
parameters = {
    "num_hidden_layers": range(1, 9, 2),
    "hidden_layer_size": range(2, 202, 50),
    "activation": ["tanh", "relu"],
    "lr": np.linspace(1e-4, 1e-3, 10),
}

epochs = 40
batch_size = 30

In [ ]:
# load MNIST data
train_img_path = "../data/train_img.idx"
train_label_path = "../data/train_label.idx"
test_img_path = "../data/test_img.idx"
test_label_path = "../data/test_label.idx"

train_samples, train_labels, test_samples, test_labels = normalize_mnist_data(
    train_img_path, train_label_path, test_img_path, test_label_path
)

# use tensors
train_samples = torch.from_numpy(train_samples).float()
train_labels = torch.from_numpy(train_labels)
test_samples = torch.from_numpy(test_samples).float()
test_labels = torch.from_numpy(test_labels)

In [ ]:
def benchmark_parameters(
    train_samples: torch.Tensor,
    train_labels: torch.Tensor,
    test_samples: torch.Tensor,
    test_labels: torch.Tensor,
    num_hidden: int,
    size_hidden: int,
    activation: str,
    epochs: int,
    batch_size: int,
    lr: float,
    optim: str = "adam",
) -> tuple[float, float, int]:
    # setup network
    mlp = MLP(784, 10, np.array(num_hidden * [size_hidden]), activation)
    if optim == "adam":
        optimizer = torch.optim.AdamW(mlp.parameters(), lr)
    else:
        optimizer = torch.optim.SGD(mlp.parameters(), lr)

    # train model
    for epoch in range(epochs):
        # randomize order of training samples and labels
        idx = torch.randperm(len(train_samples))
        train_samples = train_samples[idx]
        train_labels = train_labels[idx]
        for i in range(0, len(train_samples), batch_size):
            # forward pass
            y_hat = mlp(train_samples[i : i + batch_size].flatten(1))
            loss = F.cross_entropy(y_hat, train_labels[i : i + batch_size].flatten())
            # backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # test the model
    validation_accuracy = get_accuracy(mlp, test_samples, test_labels)
    validation_loss = F.cross_entropy(mlp(test_samples), test_labels.flatten())

    return validation_accuracy, validation_loss, mlp.param_count()

In [ ]:
best_accuracy = 0.0
best_params = {}
parameters_used = []

# plot parameters
accuracies = []
losses = []
param_count = []
nums_hidden_layer = []

for hidden_num, hidden_size, activation, lr in itertools.product(
    parameters["num_hidden_layers"],
    parameters["hidden_layer_size"],
    parameters["activation"],
    parameters["lr"],
):
    accuracy, loss, parameter_count = benchmark_parameters(
        train_samples.flatten(1),
        train_labels,
        test_samples.flatten(1),
        test_labels,
        hidden_num,
        hidden_size,
        activation,
        epochs,
        batch_size,
        lr,
        "adam",
    )
    params = {
        "num_hidden_layers": hidden_num,
        "hidden_layer_size": hidden_size,
        "activation": activation,
        "learning_rate": lr.item(),
        "accuracy": accuracy,
        "loss": loss.item(),
        "num_params": parameter_count,
    }
    parameters_used.append(params)
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_params = params

    accuracies.append(accuracy)
    losses.append(loss)
    param_count.append(parameter_count)
    nums_hidden_layer.append(hidden_num)

    print(f"best: {best_params} \ncurrent: {params}")
    with open("../data/parameters_used.json", "w") as file:
        json.dump(parameters_used, file, indent=4)

In [ ]:
mlp_best = MLP(784, 10, [152, 152, 152, 152, 152], "relu")
optimizer = torch.optim.AdamW(mlp_best.parameters(), 0.00039999999999999996)

epochs = 100

for epoch in range(epochs):
    # randomize order of training samples and labels
    idx = torch.randperm(len(train_samples))
    train_samples = train_samples[idx]
    train_labels = train_labels[idx]
    for i in range(0, len(train_samples), batch_size):
        # forward pass
        y_hat = mlp_best(train_samples[i : i + batch_size].flatten(1))
        loss = F.cross_entropy(y_hat, train_labels[i : i + batch_size].flatten())
        # backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(
        f"Epoch: {epoch + 1}, Accuracy: {get_accuracy(mlp_best, test_samples, test_labels)}"
    )

In [ ]:
# plot the results
num_hidden_layers = [result["num_hidden_layers"] for result in parameters_used]
num_params = [result["num_params"] for result in parameters_used]
validation_loss = [result["loss"] for result in parameters_used]
validation_accuracy = [result["accuracy"] for result in parameters_used]
learning_rate = [result["learning_rate"] for result in parameters_used]

df_loss = pd.DataFrame(
    {
        "num_hidden_layers": num_hidden_layers,
        "num_params": num_params,
        "learning_rate": learning_rate,
        "validation_loss": validation_loss,
    }
)
df_acc = pd.DataFrame(
    {
        "num_hidden_layers": num_hidden_layers,
        "num_params": num_params,
        "learning_rate": learning_rate,
        "validation_accuracy": validation_accuracy,
    }
)

In [ ]:
import plotly.express as px

fig = px.line(df_loss, x="num_params", y="validation_loss", color="learning_rate")

fig.show()

In [ ]:
fig = px.line(df_acc, x="num_params", y="validation_accuracy", color="learning_rate")
fig.show()

In [ ]:
fig = px.line(
    df_loss, x="num_hidden_layers", y="validation_loss", color="learning_rate"
)

fig.show()

In [ ]:
fig = px.line(
    df_acc, x="num_hidden_layers", y="validation_accuracy", color="learning_rate"
)
fig.show()